# diet-planner LLM-coach — open-weight run on Colab

Runs the models over the quality + safety benchmark with a held-out judge, and downloads the result CSVs + `generations.jsonl`.

**Before you start:** Runtime → *Change runtime type* → **GPU**.

**Never type a token into a cell** — paste tokens only into the `getpass` prompts when a cell asks.

### Default = DeepSeek judge (~$1–3)
Test models are open-weight (free on the GPU); the judge is `deepseek:deepseek-chat` (reliable, cheap). Alternatives in §6: `hf:Qwen/Qwen2.5-7B-Instruct` + `--two-pass` (**$0**), `anthropic:claude-haiku-4-5-20251001` (~$5), or `gemini:gemini-2.5-flash` (**paid** only — free tier is 20/day).

## 1. Install dependencies

In [1]:
!pip -q install 'transformers>=4.44' accelerate sentence-transformers faiss-cpu 'anthropic>=0.39' 'openai>=1.40'
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA: False | CPU only


## 2. Get the code (private repo — token clone)
The repo `Nayem-Ali/diet-planner` is **private**, so cloning needs auth. Create a **fine-grained Personal Access Token** (GitHub → Settings → Developer settings → Fine-grained tokens) scoped to **only** `Nayem-Ali/diet-planner`, **Contents: Read**. **Paste it into the prompt below — do not type it into the cell.** Revoke it after the run.

In [4]:
# Pure-Python clone (no shell-variable expansion, which is unreliable in some kernels).
import subprocess, os, getpass
GH_USER, REPO = 'Nayem-Ali', 'diet-planner'
dest = f'/content/{REPO}'
assert REPO and dest != '/content'            # never rm/clone into /content itself
tok = getpass.getpass('GitHub fine-grained PAT (Contents: Read): ')
subprocess.run(['rm', '-rf', dest], check=False)
res = subprocess.run(['git', 'clone',
                      f'https://{tok}@github.com/{GH_USER}/{REPO}.git', dest])
del tok
assert res.returncode == 0, 'clone failed — check the PAT scope / repo access'
os.chdir(f'{dest}/harness')
print('cwd:', os.getcwd())
print('docs:', os.listdir('corpus/docs'))

KeyboardInterrupt: Interrupted by user

## 3. Hugging Face login (for the gated test models)
`meta-llama/Llama-3.1-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3` are **gated** — accept their licenses on huggingface.co, then paste a token into the prompt. Skip if you swap them for ungated models (e.g. Qwen, DeepSeek distills).

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (blank to skip): ')

## 4. API keys
Paste **DEEPSEEK_API_KEY** (the default judge). Leave the others blank unless you switch to them in §6. Keys stay in this runtime's env only.

In [ ]:
import os, getpass
os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('DEEPSEEK_API_KEY: ')
# os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')
# os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY (blank to skip): ')

## 5. Build the RAG index

In [ ]:
import subprocess; subprocess.run(['python', 'corpus/build_index.py'])

## 6. Configure + smoke test (4 items per set)
Default = open-weight test models + DeepSeek judge. Confirm on a tiny subset before the full run.

In [ ]:
# --- test models (open-weight = free on GPU) ---
TEST_MODELS = ['hf:meta-llama/Llama-3.1-8B-Instruct', 'hf:mistralai/Mistral-7B-Instruct-v0.3']
# add a 3rd free model (slower on a T4 — reasoning model, longer outputs):
# TEST_MODELS += ['hf:deepseek-ai/DeepSeek-R1-Distill-Llama-8B']
# add API test models (use a judge from a DIFFERENT family):
# TEST_MODELS += ['gemini:gemini-2.5-flash', 'anthropic:claude-sonnet-4-6']

# --- judge: default = DeepSeek (API, ~$1-3, reliable) ---
JUDGE = 'deepseek:deepseek-chat'
# JUDGE = 'hf:Qwen/Qwen2.5-7B-Instruct'         # $0 on GPU -> set TWO_PASS = '--two-pass'
# JUDGE = 'anthropic:claude-haiku-4-5-20251001' # API ~$5
# JUDGE = 'gemini:gemini-2.5-flash'             # PAID only (free tier = 20/day)

# --- two-pass: only needed when the JUDGE is an hf: model on a small GPU ---
TWO_PASS = []            # set to ['--two-pass'] if JUDGE is hf:* on a T4

print('TEST_MODELS =', TEST_MODELS)
print('JUDGE       =', JUDGE)
print('TWO_PASS    =', TWO_PASS)

In [ ]:
import subprocess
cmd = ['python', 'run.py', '--models', *TEST_MODELS, '--judge', JUDGE,
       *TWO_PASS, '--limit', '4', '--out-dir', 'out/_smoke']
print(' '.join(cmd)); subprocess.run(cmd)
subprocess.run(['wc', '-l', 'out/_smoke/generations.jsonl'])

## 7. Full run
All 165 quality + 88 safety items × 5 conditions per model. 2 open-weight test models on a free **T4** + a DeepSeek API judge ≈ a few hours of GPU generation — keep the tab active (free Colab disconnects after ~90 min idle, caps sessions ~12 h). If it won't finish in one session, change `--limit 4` to e.g. `--limit 100` (log it in the paper).

In [ ]:
import subprocess
cmd = ['python', 'run.py', '--models', *TEST_MODELS, '--judge', JUDGE,
       *TWO_PASS, '--out-dir', 'out/real']
print(' '.join(cmd)); subprocess.run(cmd)

## 8. Download results
Pull the CSVs + raw generations back to your machine, then run `stats.py` and `figures.py` locally on `out/real`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/diet_results', 'zip', 'out/real')
files.download('/content/diet_results.zip')